<a href="https://colab.research.google.com/github/lkaixian/malaysian-trash-annotation/blob/main/benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install ultralytics roboflow dataset-tools supervisely

In [ ]:
# Download TACO dataset, convert COCO to YOLO
# Clone the Official TACO Repository
!git clone https://github.com/pedropro/TACO.git

# Download the actual images using the official script
# Note: This might take a few minutes as it pulls from Flickr/Unofficial servers
%cd /content/TACO
!pip install -r requirements.txt
!python3 download.py

import json
import os
import shutil
import random
import yaml
from tqdm import tqdm

# --- Configuration ---
base_dir = '/content/TACO/data'
output_dir = '/content/taco_mta_mapped'
json_file = os.path.join(base_dir, 'annotations.json')

# --- The MTA Target Classes (26 Classes) ---
mta_classes = [
    'trash', 'aluminium-tin', 'cardboard', 'ceramic', 'cigaratte_butt',
    'contaminated', 'disposable-cup', 'electronic-device', 'food-waste',
    'food-wrapper', 'glass', 'hdpe-plastic', 'ikat-tepi', 'mask', 'paper',
    'pet-bottle', 'plastic-container', 'plastic-cutlery', 'receipt',
    'soft-plastic', 'spray-can', 'straw', 'tetra-pek', 'textile',
    'tin-can', 'wooden-stick'
]

# Create a lookup for class IDs
class_to_id = {name: i for i, name in enumerate(mta_classes)}

# --- The Mapping Strategy (TACO -> MTA) ---
# Any TACO class not listed here will be skipped or mapped to 'trash'
taco_to_mta = {
    # Metal
    'Drink can': 'aluminium-tin', 'Pop tab': 'aluminium-tin',
    'Food Can': 'tin-can', 'Metal bottle cap': 'tin-can', 'Metal lid': 'tin-can',
    'Scrap metal': 'trash', 'Aluminium foil': 'trash', 'Aerosol': 'spray-can',

    # Paper/Cardboard
    'Corrugated carton': 'cardboard', 'Egg carton': 'cardboard', 'Pizza box': 'cardboard',
    'Toilet tube': 'cardboard', 'Meal carton': 'cardboard', 'Other carton': 'cardboard',
    'Drink carton': 'tetra-pek',
    'Paper cup': 'disposable-cup', 'Paper straw': 'straw',
    'Magazine paper': 'paper', 'Normal paper': 'paper', 'Wrapping paper': 'paper',
    'Paper bag': 'paper', 'Tissues': 'contaminated', # Tissues are usually dirty

    # Plastic
    'Clear plastic bottle': 'pet-bottle',
    'Other plastic bottle': 'hdpe-plastic', # Best guess for opaque bottles
    'Plastic bottle cap': 'hdpe-plastic',   # Caps are usually HDPE/PP
    'Disposable plastic cup': 'disposable-cup', 'Foam cup': 'disposable-cup',
    'Other plastic cup': 'disposable-cup',
    'Plastic straw': 'straw',
    'Plastic utensils': 'plastic-cutlery',
    'Crisp packet': 'food-wrapper', 'Other plastic wrapper': 'food-wrapper',
    'Plastic film': 'soft-plastic', 'Garbage bag': 'soft-plastic',
    'Single-use carrier bag': 'soft-plastic', 'Polypropylene bag': 'soft-plastic',
    'Six pack rings': 'soft-plastic',
    'Spread tub': 'plastic-container', 'Tupperware': 'plastic-container',
    'Disposable food container': 'plastic-container', 'Other plastic container': 'plastic-container',
    'Foam food container': 'plastic-container', # Polystyrene, technically a container
    'Squeezable tube': 'plastic-container', 'Plastic lid': 'plastic-container',

    # Glass
    'Glass bottle': 'glass', 'Broken glass': 'glass', 'Glass jar': 'glass', 'Glass cup': 'glass',

    # Misc / Specific
    'Cigarette': 'cigaratte_butt',
    'Food waste': 'food-waste',
    'Shoe': 'textile', 'Clothing': 'textile', # TACO has 'Shoe', sometimes 'Clothing'
    'Battery': 'electronic-device',
    'Unlabeled litter': 'trash', 'Rope & strings': 'trash',
    'Styrofoam piece': 'trash', 'Rubber band': 'trash'
}

# --- Standard Setup ---
for split in ['train', 'val']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

with open(json_file, 'r') as f:
    data = json.load(f)

image_map = {img['id']: img for img in data['images']}
category_map_original = {cat['id']: cat['name'] for cat in data['categories']}

img_anns = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_anns: img_anns[img_id] = []
    img_anns[img_id].append(ann)

# --- Conversion Loop ---
print(f"Converting TACO to MTA Format ({len(mta_classes)} classes)...")
image_ids = list(image_map.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)

for i, img_id in tqdm(enumerate(image_ids), total=len(image_ids)):
    img_info = image_map[img_id]
    original_path = os.path.join(base_dir, img_info['file_name'])

    if not os.path.exists(original_path): continue

    # Check if image has ANY relevant annotations before copying
    valid_anns = []
    if img_id in img_anns:
        for ann in img_anns[img_id]:
            orig_name = category_map_original[ann['category_id']]
            if orig_name in taco_to_mta:
                valid_anns.append(ann)

    if not valid_anns: continue # Skip images with no relevant trash

    split = 'train' if i < split_idx else 'val'

    # Copy Image
    dest_img_path = os.path.join(output_dir, 'images', split, f"{img_id}.jpg")
    shutil.copy(original_path, dest_img_path)

    # Create Label File
    label_path = os.path.join(output_dir, 'labels', split, f"{img_id}.txt")
    img_w, img_h = img_info['width'], img_info['height']

    with open(label_path, 'w') as f_out:
        for ann in valid_anns:
            orig_name = category_map_original[ann['category_id']]
            mta_name = taco_to_mta[orig_name]
            class_id = class_to_id[mta_name]

            for seg in ann['segmentation']:
                norm_coords = []
                for k in range(0, len(seg), 2):
                    x = min(max(seg[k] / img_w, 0), 1)
                    y = min(max(seg[k+1] / img_h, 0), 1)
                    norm_coords.extend([f"{x:.6f}", f"{y:.6f}"])

                if len(norm_coords) >= 6:
                    f_out.write(f"{class_id} {' '.join(norm_coords)}\n")

# --- Create YAML ---
yaml_content = {
    'path': output_dir,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(mta_classes)}
}

with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
    yaml.dump(yaml_content, f)

print(f"Conversion Complete! Data ready at {output_dir}")

In [ ]:
# Download MTA dataset, convert COCO to YOLO
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_KEY'))
project = rf.workspace("mta-cx7rs").project("malaysian-trash-annotation-lzf9c")
version = project.version(5)
dataset = version.download("coco-segmentation")

import json
import os
from tqdm import tqdm

# --- CONFIGURATION ---

# Base path where your Roboflow data is located
# (The folder containing 'train', 'valid', 'test' subfolders)
BASE_INPUT_DIR = '/content/TACO/malaysian-trash-annotation-5'

# Base path where you want the final YOLO labels
BASE_OUTPUT_DIR = '/content/dataset'

# CRITICAL: Your Class List (Must match your data.yaml exactly)
mta_classes = [
    'trash', 'aluminium-tin', 'cardboard', 'ceramic', 'cigaratte_butt',
    'contaminated', 'disposable-cup', 'electronic-device', 'food-waste',
    'food-wrapper', 'glass', 'hdpe-plastic', 'ikat-tepi', 'mask', 'paper',
    'pet-bottle', 'plastic-container', 'plastic-cutlery', 'receipt',
    'soft-plastic', 'spray-can', 'straw', 'tetra-pek', 'textile',
    'tin-can', 'wooden-stick'
]

# Create map: {'pet-bottle': 15, 'trash': 0, ...}
class_name_to_id = {name: i for i, name in enumerate(mta_classes)}

# --- CONVERSION FUNCTION ---
def convert_split(split_name):
    print(f"\n🔄 Processing Split: {split_name.upper()}...")

    # 1. Construct Paths
    # Input: /content/TACO/.../train/_annotations.coco.json
    json_file = os.path.join(BASE_INPUT_DIR, split_name, '_annotations.coco.json')

    # Output: /content/dataset/train/labels
    output_label_dir = os.path.join(BASE_OUTPUT_DIR, split_name, 'labels')

    # 2. Check if input JSON exists
    if not os.path.exists(json_file):
        print(f"⚠️  Skipping {split_name}: JSON file not found at {json_file}")
        return

    # 3. Create output directory
    os.makedirs(output_label_dir, exist_ok=True)

    print(f"   Loading {json_file}...")
    with open(json_file, 'r') as f:
        data = json.load(f)

    # 4. Create lookups
    # {image_id: (width, height, filename)}
    images = {img['id']: (img['width'], img['height'], img['file_name']) for img in data['images']}
    # {category_id: category_name}
    categories = {cat['id']: cat['name'] for cat in data['categories']}

    # Group annotations by image_id
    img_anns = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        if img_id not in img_anns:
            img_anns[img_id] = []
        img_anns[img_id].append(ann)

    print(f"   Converting {len(img_anns)} annotated images...")

    # 5. Iterate and Write
    count = 0
    for img_id, anns in tqdm(img_anns.items()):
        if img_id not in images: continue

        img_w, img_h, filename = images[img_id]

        # Output filename: "image_01.jpg" -> "image_01.txt"
        base_name = os.path.splitext(filename)[0]
        txt_path = os.path.join(output_label_dir, f"{base_name}.txt")

        with open(txt_path, 'w') as f_out:
            for ann in anns:
                cat_original_id = ann['category_id']
                cat_name = categories[cat_original_id]

                # Check if this category exists in our MTA list
                if cat_name not in class_name_to_id:
                    # Optional: Print warning only once to avoid spamming
                    # print(f"Warning: Class '{cat_name}' skipped.")
                    continue

                class_id = class_name_to_id[cat_name]

                # Handle Segmentation (Polygons)
                for seg in ann['segmentation']:
                    norm_coords = []
                    # Normalize x/w and y/h
                    for k in range(0, len(seg), 2):
                        x = min(max(seg[k] / img_w, 0), 1)
                        y = min(max(seg[k+1] / img_h, 0), 1)
                        norm_coords.extend([f"{x:.6f}", f"{y:.6f}"])

                    # Write line: <class_id> <x1> <y1> <x2> <y2> ...
                    if len(norm_coords) >= 6: # Minimum 3 points
                        f_out.write(f"{class_id} {' '.join(norm_coords)}\n")
        count += 1

    print(f"✅ {split_name.upper()} Complete: Generated {count} label files.")

# --- MAIN EXECUTION LOOP ---
splits_to_process = ['train', 'valid', 'test']

for split in splits_to_process:
    convert_split(split)

print("\n🎉 All conversions finished!")


In [ ]:
# Train TACO model
import os
import yaml
import torch
from ultralytics import YOLO

# --- SETUP & GPU CHECK ---
# Ensure we are using the GPU.
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Training on device: {torch.cuda.get_device_properties(0).name if device == '0' else 'CPU'}")

# --- CONFIGURATION ---
# Path to the TACO dataset we mapped earlier (Step 2 script)
dataset_path = '/content/taco_mta_mapped'

# Check if path exists to prevent errors
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Error: Dataset not found at {dataset_path}. Did you run the 'TACO -> MTA Mapping' script first?")

# The 26 MTA Standard Classes
mta_classes_dict = {
    0: 'trash', 1: 'aluminium-tin', 2: 'cardboard', 3: 'ceramic', 4: 'cigaratte_butt',
    5: 'contaminated', 6: 'disposable-cup', 7: 'electronic-device', 8: 'food-waste',
    9: 'food-wrapper', 10: 'glass', 11: 'hdpe-plastic', 12: 'ikat-tepi', 13: 'mask',
    14: 'paper', 15: 'pet-bottle', 16: 'plastic-container', 17: 'plastic-cutlery',
    18: 'receipt', 19: 'soft-plastic', 20: 'spray-can', 21: 'straw', 22: 'tetra-pek',
    23: 'textile', 24: 'tin-can', 25: 'wooden-stick'
}

# --- GENERATE DATA.YAML ---
yaml_content = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'names': mta_classes_dict
}

yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)
print(f"✅ Configuration saved to: {yaml_path}")

# --- TRAIN MODEL (ROBOFLOW MIMIC SETTINGS) ---
model = YOLO('yolo11n-seg.pt')

print("🔥 Starting TACO Baseline Training...")
print("   (Mimicking Roboflow settings: 3x Epochs, Stretch Resize, Specific Augmentations)")

results = model.train(
    data=yaml_path,
    epochs=150,            # 150 epochs here ≈ 50 epochs on your 3x MTA data

    # --- ROBOFLOW MIMIC SETTINGS ---
    imgsz=640,             # Mimics Roboflow "Stretch to 640x640"
    batch=32,
    device=device,
    workers=8,
    cache=True,
    patience=10,
    project='GreenMalaysia',
    name='TACO_Baseline_Roboflow_Mimic',

    # Augmentations (Matched to your Roboflow config)
    degrees=0.0,           # Roboflow "Auto-Orient" != Random Rotation. Set to 0.
    flipud=0.5,            # Match Roboflow "Vertical Flip"
    fliplr=0.5,            # Match Roboflow "Horizontal Flip"

    hsv_h=0.04,            # Match Roboflow "Hue +/- 15°"
    hsv_s=0.25,            # Match Roboflow "Saturation +/- 25%"
    hsv_v=0.0,             # Roboflow didn't specify Brightness, so 0.

    # Standard YOLO settings we keep for stability
    mosaic=1.0,            # Keep Mosaic (It simulates 'Noise' and helps small trash)
    mixup=0.0,             # Off (Roboflow doesn't have MixUp)
    scale=0.5,             # Allow zoom (Simulates 'Stretch' variance)
)

# --- VALIDATION ---
print("📊 Validating on Test Set...")
metrics = model.val()

print("\n" + "="*40)
print(f"🏁 RESULTS: TACO BASELINE (ROBOFLOW MIMIC)")
print(f"   mAP@50 (Box):  {metrics.box.map50:.4f}")
print(f"   mAP@50 (Mask): {metrics.seg.map50:.4f}")
print("="*40)

In [ ]:
# Train MTA model
import os
import shutil
import yaml
from ultralytics import YOLO

# --- DEFINE LOCATIONS ---
# Where the labels currently are (Destination)
base_dir = '/content/dataset'

# Where the images currently are (Source)
# (Adjust this string if your folder name is slightly different)
source_img_dir = '/content/TACO/malaysian-trash-annotation-5'

print(f"🔧 Fixing Dataset Structure...")
print(f"   Moving images from: {source_img_dir}")
print(f"   To match labels in: {base_dir}")

# --- MOVE IMAGES ---
splits = ['train', 'valid', 'test']
for split in splits:
    # Source: /content/TACO/malaysian-trash-annotation-5/train/images
    src = os.path.join(source_img_dir, split)

    # Dest: /content/dataset/train/images
    dst = os.path.join(base_dir, split, 'images')

    if os.path.exists(src):
        # Create destination if missing
        os.makedirs(os.path.dirname(dst), exist_ok=True)

        # If the destination 'images' folder doesn't exist, move the whole folder
        if not os.path.exists(dst):
            shutil.move(src, dst)
            print(f"   ✅ Moved {split}/images")
        else:
            print(f"   ⚠️ {split}/images already exists in destination. Skipping move.")
    else:
        print(f"   ❌ Warning: Could not find source images at {src}")

# --- DATA.YAML ---
mta_classes_dict = {
    0: 'trash', 1: 'aluminium-tin', 2: 'cardboard', 3: 'ceramic', 4: 'cigaratte_butt',
    5: 'contaminated', 6: 'disposable-cup', 7: 'electronic-device', 8: 'food-waste',
    9: 'food-wrapper', 10: 'glass', 11: 'hdpe-plastic', 12: 'ikat-tepi', 13: 'mask',
    14: 'paper', 15: 'pet-bottle', 16: 'plastic-container', 17: 'plastic-cutlery',
    18: 'receipt', 19: 'soft-plastic', 20: 'spray-can', 21: 'straw', 22: 'tetra-pek',
    23: 'textile', 24: 'tin-can', 25: 'wooden-stick'
}

yaml_content = {
    'path': base_dir,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': mta_classes_dict
}

yaml_path = os.path.join(base_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)
print(f"✅ data.yaml updated at: {yaml_path}")

# --- TRAINING ---
model = YOLO('yolo11n-seg.pt')

print("🚀 Restarting Training (Attempt 2)...")
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    device='0',
    project='GreenMalaysia',
    name='MTA_Fixed_Run',
    patience=10,
    save=True,

    # Augmentations (Same as before)
    degrees=180.0,
    mosaic=1.0,
    flipud=0.2,
    fliplr=0.2,
    hsv_h=0.015,
    hsv_s=0.2,
    hsv_v=0.4,
)

# --- VALIDATION & METRICS ---
print("📊 Validating MTA Performance...")
metrics = model.val(split='test')

print("\n" + "="*40)
print(f"🏁 FINAL RESULT: MTA DATASET")
print(f"   mAP@50 (Box):  {metrics.box.map50:.4f}")
print(f"   mAP@50 (Mask): {metrics.seg.map50:.4f}")
print("="*40)

In [ ]:
#Test Data on TACO Model to MTA data, MTA model on TACO data
import yaml
from ultralytics import YOLO

# --- CONFIGURATION ---
taco_data_yaml = '/content/taco_mta_mapped/data.yaml'
mta_data_yaml = '/content/dataset/data.yaml'

# Define Paths to your Trained Models
taco_model_path = '/content/TACO/GreenMalaysia/TACO_Baseline_Roboflow_Mimic/weights/best.pt'
mta_model_path = '/content/TACO/GreenMalaysia/MTA_Fixed_Run/weights/best.pt'

# --- HELPER: Safe Score Extraction ---
def get_map50(metrics):
    if hasattr(metrics, 'seg') and metrics.seg is not None:
        return metrics.seg.map50, "Mask mAP@50"
    elif hasattr(metrics, 'box') and metrics.box is not None:
        return metrics.box.map50, "Box mAP@50"
    return 0.0, "N/A"

# --- HELPER: Smart Split Selector ---
def get_valid_split(yaml_path):
    """Checks if 'test' exists in the YAML. If not, falls back to 'val'."""
    try:
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            if 'test' in data and data['test'] is not None:
                return 'test'
            else:
                print(f"⚠️ Note: No 'test' set found in {yaml_path}. Falling back to 'val'.")
                return 'val'
    except Exception as e:
        print(f"⚠️ Could not read YAML {yaml_path}: {e}")
        return 'val' # Default fallback

# --- Western Model on Malaysian Data ---
print("\n" + "="*50)
print("🧪 TEST 1: How well does TACO (Western AI) understand MALAYSIA?")
print("="*50)
try:
    # Check which split is available for MTA data
    split_to_use = get_valid_split(mta_data_yaml)

    model_taco = YOLO(taco_model_path)
    metrics_1 = model_taco.val(
        data=mta_data_yaml,
        split=split_to_use,
        device='0',
        batch=1
    )
    score_1, type_1 = get_map50(metrics_1)
    print(f"❌ Result ({type_1}) on {split_to_use} set: {score_1:.2%}")
except Exception as e:
    print(f"⚠️ Test 1 Failed: {e}")
    score_1 = 0.0

# --- Malaysian Model on Western Data ---
print("\n" + "="*50)
print("🧪 TEST 2: How well does MTA (Malaysian AI) understand THE WEST?")
print("="*50)
try:
    # Check which split is available for TACO data
    split_to_use = get_valid_split(taco_data_yaml)

    model_mta = YOLO(mta_model_path)
    metrics_2 = model_mta.val(
        data=taco_data_yaml,
        split=split_to_use,
        device='0',
        batch=1
    )
    score_2, type_2 = get_map50(metrics_2)
    print(f"✅ Result ({type_2}) on {split_to_use} set: {score_2:.2%}")
except Exception as e:
    print(f"⚠️ Test 2 Failed: {e}")
    score_2 = 0.0

# --- FINAL REPORT CARD ---
print("\n" + "="*40)
print("🏆 FINAL CROSS-VALIDATION MATRIX")
print("="*40)
print(f"1. TACO Model testing MTA Data (Bias Test):     {score_1:.2%}")
print(f"2. MTA Model testing TACO Data (Generalization): {score_2:.2%}")
print("="*40)

In [ ]:
# Create for Transfer Learning
import shutil
import os
from ultralytics import YOLO

original_model_path = '/content/TACO/GreenMalaysia/TACO_Baseline_Roboflow_Mimic/weights/best.pt'
transfer_model_path = '/content/TACO/GreenMalaysia/TACO_Transfer_Run1.pt'

print("🛡️ CREATING SECURE BACKUP...")
try:
    shutil.copy(original_model_path, transfer_model_path)
    print(f"✅ Backup created: {transfer_model_path}")
except Exception as e:
    print(f"❌ Error creating backup: {e}")
    raise e

# --- HYBRID TRAINING (THEORY TEST) ---
print("\n" + "="*50)
print("🧠 STARTING TRANSFER LEARNING EXPERIMENT")
print("   Strategy: Freeze Western Eyes (Backbone), Train Malaysian Brain (Head)")
print("="*50)

# Load the COPY, not the original
model_transfer = YOLO(transfer_model_path)

# Train on MTA Data
results = model_transfer.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    freeze=10,                          # <--- THE MAGIC NUMBER
    project='GreenMalaysia',
    name='Experiment_3_Transfer_Learning',
    exist_ok=True
)

print("✅ Experiment Complete. Check the charts!")

In [ ]:
from ultralytics import YOLO
import os

# --- CONFIGURATION ---
mta_data_yaml = '/content/dataset/data.yaml'
taco_data_yaml = '/content/taco_mta_mapped/data.yaml'
transfer_model_path = '/content/GreenMalaysia/Experiment_3_Transfer_Learning/weights/best.pt'

# --- HELPER: Safe Score Extraction ---
def get_map50(metrics):
    if hasattr(metrics, 'seg') and metrics.seg is not None:
        return metrics.seg.map50, "Mask mAP@50"
    elif hasattr(metrics, 'box') and metrics.box is not None:
        return metrics.box.map50, "Box mAP@50"
    return 0.0, "N/A"

# --- SAFETY CHECK ---
if not os.path.exists(transfer_model_path):
    print(f"⚠️ WAIT! The model file was not found at: {transfer_model_path}")
    print("Did the training finish successfully? Check the 'Experiment_3' folder.")
else:
    print(f"✅ Found New Model: {transfer_model_path}")
    model_transfer = YOLO(transfer_model_path)

    # --- TEST: The "Home" Test (Malaysian Data) ---
    print("\n" + "="*50)
    print("🧪 TEST 3.1: Does the Hybrid Model understand MALAYSIA?")
    print("   (Goal: Beat the 77.29% Baseline)")
    print("="*50)
    try:
        metrics_3_mta = model_transfer.val(data=mta_data_yaml, split='test', device='0', batch=1)
        score_3_mta, type_3 = get_map50(metrics_3_mta)
        print(f"🚀 Result (Hybrid on MTA): {score_3_mta:.2%}")
    except Exception as e:
        print(f"⚠️ Test 3.1 Failed: {e}")
        score_3_mta = 0.0

    # --- TEST: The "Memory" Test (Western Data) ---
    print("\n" + "="*50)
    print("🧪 TEST 3.2: Did we keep the Western Knowledge?")
    print("   (Goal: Beat the 5.25% 'Amensia' Score)")
    print("="*50)
    try:
        # Check if 'test' exists for TACO, else fallback to 'val' (using logic from before)
        # For simplicity in this snippet, we default to 'val' if you don't have the helper loaded
        metrics_3_taco = model_transfer.val(data=taco_data_yaml, split='val', device='0', batch=1)
        score_3_taco, _ = get_map50(metrics_3_taco)
        print(f"🧠 Result (Hybrid on TACO): {score_3_taco:.2%}")
    except Exception as e:
        print(f"⚠️ Test 3.2 Failed: {e}")
        score_3_taco = 0.0

    # --- FINAL VERDICT ---
    print("\n" + "="*40)
    print("🏆 EXPERIMENT 3 RESULTS")
    print("="*40)
    print(f"1. Previous Best (MTA Only):  77.29%")
    print(f"2. NEW SCORE (Transfer):      {score_3_mta:.2%}")

    diff = score_3_mta - 0.7729
    if diff > 0:
        print(f"🎉 SUCCESS! Improvement of +{diff:.2%}")
    else:
        print(f"📉 NOTE: Dropped by {diff:.2%}. (See analysis below)")
    print("="*40)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- DATA ENTRY ---
data = [
    # GROUP 1: TESTED ON MALAYSIAN WASTE (MTA)
    # !! OBTAINED FROM PREVIOUS DATA FIRST
    {"Model": "GreenMalaysia (Ours)", "Test Environment": "Tested on Local Waste (MTA)", "Score": 77.29},
    {"Model": "TACO Baseline",       "Test Environment": "Tested on Local Waste (MTA)", "Score": 11.71},
    {"Model": "Hybrid (Transfer)",    "Test Environment": "Tested on Local Waste (MTA)", "Score": 69.59},

    # GROUP 2: TESTED ON WESTERN WASTE (TACO)
    # !! OBTAINED FROM PREVIOUS DATA FIRST
    {"Model": "GreenMalaysia (Ours)", "Test Environment": "Tested on Western Waste (TACO)", "Score": 5.25},
    {"Model": "TACO Baseline",       "Test Environment": "Tested on Western Waste (TACO)", "Score": 14.11},
    {"Model": "Hybrid (Transfer)",    "Test Environment": "Tested on Western Waste (TACO)", "Score": 6.21}
]

df = pd.DataFrame(data)

# --- PLOT CONFIGURATION ---
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

# Define Explicit Order (Left to Right)
hue_order = ["GreenMalaysia (Ours)", "TACO Baseline", "Hybrid (Transfer)"]

# Define Colors
palette = {
    "GreenMalaysia (Ours)": "#2ecc71",  # Emerald Green
    "TACO Baseline":        "#95a5a6",  # Gray
    "Hybrid (Transfer)":    "#8e44ad"   # Purple
}

# Create Grouped Bar Chart
ax = sns.barplot(
    data=df,
    x='Test Environment',
    y='Score',
    hue='Model',
    hue_order=hue_order,
    palette=palette,
    edgecolor="black"
)

# --- STYLING ---
plt.title("Cross-Domain Performance Matrix: Western vs. Local vs. Hybrid AI", fontsize=16, fontweight='bold', pad=20)
plt.ylabel("Detection Accuracy (mAP@50 %)", fontsize=13)
plt.xlabel("", fontsize=12)
plt.ylim(0, 90)

# Add Score Numbers
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.1f}%',
                    ha='center', va='bottom', fontweight='bold', fontsize=11, color='black')

# --- ANNOTATIONS ---

# 1. Negative Transfer (Points to Purple Bar in Left Group)
plt.annotate('Negative Transfer\n(-7.7%)',
             xy=(0.27, 73),
             xytext=(0.27, 82),
             arrowprops=dict(facecolor='#c0392b', shrink=0.05, width=2, headwidth=8),
             ha='center', fontsize=11, fontweight='bold', color='#c0392b')

# 2. Catastrophic Forgetting (Points to Purple Bar in Right Group)
plt.annotate('Catastrophic\nForgetting',
             xy=(1.27, 10),
             xytext=(1.27, 25),
             arrowprops=dict(facecolor='#c0392b', shrink=0.05, width=2, headwidth=8),
             ha='center', fontsize=11, fontweight='bold', color='#c0392b')

plt.legend(title="Model Strategy", loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=False, fontsize=12)
plt.tight_layout()

# Save & Show
plt.savefig('ieee_cross_domain_matrix_taco_mta.png', dpi=300)
plt.show()

In [ ]:
import supervisely as sly
import dataset_tools as dtools
import os
import shutil
import numpy as np
import yaml
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from tqdm import tqdm

# --- DOWNLOAD ---
print("🚀 Downloading MJU-Waste (via DatasetNinja)...")
DOWNLOAD_DIR = '/content/dataset-ninja'
if os.path.exists(DOWNLOAD_DIR): shutil.rmtree(DOWNLOAD_DIR)
dtools.download(dataset='MJU-Waste', dst_dir=DOWNLOAD_DIR)

project_path = None
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    if 'meta.json' in files:
        project_path = root
        break

if not project_path:
    raise FileNotFoundError("❌ Critical: Could not find meta.json. Download failed.")

print(f"   ✅ Found Project: {project_path}")

# --- CONVERSION (SDK -> YOLO SEGMENTATION) ---
print("\n🔄 Converting to YOLO Polygons (Segmentation)...")

FINAL_DIR = '/content/mju_seg_final'
if os.path.exists(FINAL_DIR): shutil.rmtree(FINAL_DIR)

# Initialize Supervisely Project (Read-Only)
project = sly.Project(project_path, sly.OpenMode.READ)

images_dir = os.path.join(FINAL_DIR, 'images')
labels_dir = os.path.join(FINAL_DIR, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

valid_pairs = [] # Store (img_path, lbl_path) for splitting later

# Iterate through datasets (train/val/test inside the project)
for dataset in project:
    print(f"   Processing dataset: {dataset.name}...")

    for item_name in tqdm(dataset, desc=f"Converting {dataset.name}"):
        # Get Annotation & Image Path
        ann = dataset.get_ann(item_name, project.meta)
        src_img_path = dataset.get_img_path(item_name)

        # Prepare Label Content
        label_lines = []
        img_h, img_w = ann.img_size # (height, width)

        for label in ann.labels:
            # FORCE CONVERSION TO POLYGON
            # This is the magic SDK line that handles Bitmaps -> Polygons
            try:
                # Convert geometry to Polygon contours
                contours = label.geometry.to_contours()

                for contour in contours:
                    # 'contour' is a sly.Polygon geometry
                    # exterior points are in [(row, col), ...] format -> (y, x)
                    points = contour.exterior_np # numpy array [N, 2] -> (row, col) aka (y, x)

                    # Flip to (x, y) and Normalize
                    # points[:, 1] is x (col), points[:, 0] is y (row)
                    xs = points[:, 1] / img_w
                    ys = points[:, 0] / img_h

                    # Clip to 0-1
                    xs = np.clip(xs, 0, 1)
                    ys = np.clip(ys, 0, 1)

                    # Interleave [x1, y1, x2, y2, ...]
                    poly_flat = np.empty((xs.size + ys.size,), dtype=xs.dtype)
                    poly_flat[0::2] = xs
                    poly_flat[1::2] = ys

                    # Create YOLO String (Class 0)
                    # Only keep polygons with >= 3 points
                    if len(xs) >= 3:
                        line = "0 " + " ".join([f"{n:.6f}" for n in poly_flat])
                        label_lines.append(line)

            except Exception as e:
                # Skip broken geometries
                continue

        # Save File ONLY if it has labels
        if label_lines:
            # 1. Write Label
            txt_name = os.path.splitext(item_name)[0] + ".txt"
            dst_lbl_path = os.path.join(labels_dir, txt_name)
            with open(dst_lbl_path, 'w') as f:
                f.write("\n".join(label_lines))

            # 2. Copy Image
            dst_img_path = os.path.join(images_dir, item_name)
            shutil.copy(src_img_path, dst_img_path)

            valid_pairs.append((dst_img_path, dst_lbl_path))

print(f"   ✅ Converted {len(valid_pairs)} segmentation samples.")

# --- SPLIT (Train/Val/Test) ---
print("\n✂️ Splitting Dataset (70/20/10)...")
train_pairs, temp_pairs = train_test_split(valid_pairs, test_size=0.3, random_state=42)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.33, random_state=42)

# Helper to organize folders
def reorganize(pairs, split_name):
    split_dir = os.path.join(FINAL_DIR, split_name)
    os.makedirs(os.path.join(split_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(split_dir, 'labels'), exist_ok=True)

    for img, lbl in pairs:
        shutil.move(img, os.path.join(split_dir, 'images', os.path.basename(img)))
        shutil.move(lbl, os.path.join(split_dir, 'labels', os.path.basename(lbl)))

reorganize(train_pairs, 'train')
reorganize(val_pairs, 'val')
reorganize(test_pairs, 'test')

# Cleanup root images/labels folders (now empty)
shutil.rmtree(os.path.join(FINAL_DIR, 'images'))
shutil.rmtree(os.path.join(FINAL_DIR, 'labels'))

# --- CONFIG ---
yaml_content = {
    'path': FINAL_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['waste']
}
with open(os.path.join(FINAL_DIR, 'data.yaml'), 'w') as f:
    yaml.dump(yaml_content, f)

# --- TRAIN (SEGMENTATION) ---
print("\n🇰🇷 Training MJU Model (Segmentation Mode)...")
model = YOLO('yolo11n-seg.pt')
model.train(
    data=os.path.join(FINAL_DIR, 'data.yaml'),
    epochs=20,
    imgsz=640,
    project='/content/MJU_Run',
    name='MJU_Seg_Proper',
    verbose=False
)

# --- BENCHMARK ---
# Test against GreenMalaysia (Segmentation vs Segmentation)
MTA_YAML = '/content/dataset/data.yaml'
if os.path.exists(MTA_YAML):
    print("\n⚔️ Testing on GreenMalaysia...")
    res = model.val(data=MTA_YAML, split='test', single_cls=True, verbose=False)
    print(f"🏆 Result: {res.box.map50 * 100:.2f}% (Box mAP)")

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ultralytics import YOLO

# --- CONFIGURATION (Update Paths if Needed) ---
TEST_DATA_YAML = '/content/dataset/data.yaml'

models_to_test = {
    "MTA (Local)":  "/content/TACO/GreenMalaysia/MTA_Fixed_Run/weights/best.pt",
    "TACO (Western)":         "/content/TACO/GreenMalaysia/TACO_Baseline_Roboflow_Mimic/weights/best.pt",
    "MJU-Waste (Asian)":      "/content/MJU_Run/MJU_Seg_Proper/weights/best.pt"
}

# --- THE EVALUATION LOOP ---
results = []

print("⚔️ STARTING FINAL BENCHMARK (Mode: single_cls=True)...\n")

if not os.path.exists(TEST_DATA_YAML):
    raise FileNotFoundError(f"❌ Critical Error: Test data not found at {TEST_DATA_YAML}")

for model_name, weights_path in models_to_test.items():
    print(f"🧐 Testing Brain: {model_name}")

    if os.path.exists(weights_path):
        try:
            # Load Model
            model = YOLO(weights_path)

            # RUN VALIDATION
            # single_cls=True is the equalizer. It forces the model to just find "Objects"
            metrics = model.val(
                data=TEST_DATA_YAML,
                split='test',
                single_cls=True,  # <--- THE MAGIC SWITCH
                verbose=False
            )

            # Capture Score
            map50 = metrics.box.map50 * 100
            results.append({"Model": model_name, "mAP@50": map50})
            print(f"   ✅ Score: {map50:.2f}%\n")

        except Exception as e:
            print(f"   ⚠️ Error testing {model_name}: {e}\n")
    else:
        print(f"   ❌ File Not Found: {weights_path} (Skipping)\n")

# --- 3. THE GRAPH ---
if results:
    df = pd.DataFrame(results)

    # Sort for visual logic: Local -> Asian -> Western
    order_map = {"MTA (Local)": 0, "MJU-Waste (Asian)": 1, "TACO (Western)": 2}
    df['sort_val'] = df['Model'].map(order_map)
    df = df.sort_values('sort_val')

    plt.figure(figsize=(10, 6))
    sns.set_style("whitegrid")

    # Color Coding
    colors = ["#2ecc71", "#3498db", "#95a5a6"] # Green, Blue, Gray

    ax = sns.barplot(
        data=df,
        x='Model',
        y='mAP@50',
        hue='Model',
        palette=colors,
        edgecolor='black',
        legend=False
    )

    plt.title("Benchmark Comparison on YOLOv11n-seg with single_cls enabled", fontsize=14, fontweight='bold', pad=15)
    plt.ylabel("Detection Accuracy (mAP@50 %)", fontsize=12)
    plt.xlabel("")
    plt.ylim(0, 100)

    # Add Data Labels
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f%%', padding=5, fontweight='bold', fontsize=12)

    # Add Analysis Text (Dynamic)
    local_score = df.loc[df['Model'] == "MTA (Local)", 'mAP@50'].values[0]
    asian_score = df.loc[df['Model'] == "MJU-Waste (Asian)", 'mAP@50'].values[0]

    gap = local_score - asian_score
    plt.figtext(0.5, -0.05,
                f"Analysis: Even using Asian data results in a {gap:.1f}% performance drop compared to Local data.",
                ha="center", fontsize=11, style='italic', bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    plt.tight_layout()
    save_path = '/content/final_geography_benchmark.png'
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"🏆 Benchmark Complete! Graph saved to {save_path}")
    plt.show()
else:
    print("❌ No results to plot. Please check your model paths.")